# Logistics Operations Dashboard

Plotly visualisations for fictional logistics shipment data. Run `python src/generate_data.py` and `python src/pipeline.py` before opening this notebook.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "clean_shipments.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["rfq_sent_at", "quote_received_at"])
df.head()

## Quote Response Time Histogram

In [ ]:
fig = px.histogram(
    df,
    x="quote_response_hours",
    nbins=50,
    title="Quote Response Time Distribution",
    labels={"quote_response_hours": "Quote response hours"},
)
fig.add_vline(x=24, line_dash="dash", annotation_text="24h target")
fig.show()

## Customs Status Mix

In [ ]:
fig = px.pie(
    df,
    names="customs_status",
    title="Customs Status Breakdown",
)
fig.show()

## Fulfilment Rate By Transport Mode

In [ ]:
fulfilment_by_mode = (
    df.assign(on_time=df["fulfilment_status"].eq("on_time"))
    .groupby("transport_mode", as_index=False)["on_time"]
    .mean()
)
fulfilment_by_mode["on_time_rate_pct"] = fulfilment_by_mode["on_time"] * 100

fig = px.bar(
    fulfilment_by_mode,
    x="transport_mode",
    y="on_time_rate_pct",
    title="Fulfilment On-Time Rate By Transport Mode",
    labels={"transport_mode": "Transport mode", "on_time_rate_pct": "On-time rate (%)"},
)
fig.update_yaxes(range=[0, 100])
fig.show()

## Supplier Compliance Trend

In [ ]:
compliance_trend = (
    df.assign(
        rfq_week=df["rfq_sent_at"].dt.to_period("W").dt.start_time,
        compliant=df["supplier_compliance_flag"].astype(bool),
    )
    .groupby("rfq_week", as_index=False)["compliant"]
    .mean()
)
compliance_trend["compliance_rate_pct"] = compliance_trend["compliant"] * 100

fig = px.line(
    compliance_trend,
    x="rfq_week",
    y="compliance_rate_pct",
    markers=True,
    title="Supplier Compliance Trend Over Time",
    labels={"rfq_week": "RFQ week", "compliance_rate_pct": "Compliance rate (%)"},
)
fig.update_yaxes(range=[0, 100])
fig.show()